# Flujo de información para generar capa comunal
#### Modelo de estimación de demanda eléctrica MERLIN_EDM
--- 
En este Notebook se empleará el modelo ``merlin_edm`` entrenado pra obtener la construcción de capas geoespaciales de demanda eléctrica en escala comunal y desagregada por sector de interés. 

Los archivos de entrada para esto serán: 

- ``../data/interim/shares_comunales.parquet``: Los shares comunales provenientes de unir los datos de facturación de clientes regulados, con la venta de energía a clientes libres. Este archivo se encuentra en formato ancho, es decir, tiene: ``"año_ms" | "año" | "region"| "comuna" | f"consumo_{sector}_MWh" for sector in SECTORES | "consumo_total_mes_MWh"``
- ``../data/rec_2024_2025/temperatura_comunal_2024_2025.parquet``: Es la temperatura de los años 2024 y 2025 en todo Chile, en escala comunal y resolución horaria. 
- ``../data/raw/reg_alias.json``: Son los alias de formato ISO de la región con respecto a su nombre disponible en las bases geoespaciales. Sirve para cruzar la información del balance regional de energía con el de las temperaturas. 

El procesamiento de estos archivos de entrada llevarán a lo siguiente: 

- Generación de rezagos temporales de temperatura (temperatura presente y 7 lags hacia atrás).
- Cálculo de series trigonométricas de hora/semana/año. 
- Condicionales de día hábil/fin de semana/feriado en escala regional. 
- Intensidades energéticas en escala mensual ``total_comunal/total_nacional`` y por sector ``total_sector_comuna/total_comunal``.

Se debería generar una matriz de inputs que contenga los datos de todo el país para poder tomarlo como inferencia del modelo. El orden de los inputs importa para la red neuronal. Este se guarda en un archivo de texto llamado ``../data/rec_2024_2025/columns.txt`` y para cargarlo a la sesión el código es: 

```python
columns = []  # Lista vacía para que se guarden los nombres de las columnas
with open("../data/rec_2024_2025/columns.txt", "r") as f:
    for line in f: 
        columns.append(line.strip()) 

```

La matriz de inputs se usa para el modelo de red neuronal entrenado. Los outputs que se deberían obtener son: 

- ``../data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal.gpkg``: Geocapa con los totales anuales (año 2024 y 2025) en cada comuna, total y por sector (RCPIT)
- ``../data/rec_2024_2025/results/capas_comunales/wp2_output_demanda_electrica_comunal_ts.parquet``: Serie de tiempo con la demanda de electricidad (año 2024 y 2025) en cada comuna, total y por sector (RCPIT)

Hay que hacer un pequeño procesamiento previo para hacer la inferencia, y que corresponde a extrapolar los consumos mensuales comunales a otras comunas faltantes. Luego de la generación de la base ``shares_comunales.parquet``, quedó información de desagregación de consumos por sector en 212 de 345 comunas. Para hacer la extrapolación de esos consumos, se considerará el tamaño y cercanía de las comunas. Entonces, dada una comuna $j$ sin información de consumos, se buscará la vecina $i$ más cercana que presente área similar y se copiarán sus consumos. O bien, se calculará el promedio de consumos de todos sus vecinos.

In [12]:
# ==========================================
# CELDA 1: CONFIGURACIÓN E IMPORTACIONES
# ==========================================
import os
import sys
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
import numpy as np
import matplotlib.pyplot as plt
import holidays
import calendar
import geopandas as gpd
import warnings
import unicodedata

# 1. Configuración de Rutas Globales
BASE_DIR = os.path.abspath("..")
MODEL_PATH = os.path.join(BASE_DIR, "models", "ds_comunal", "best_merlin_mlp_global.keras")
LAYER_PATH = os.path.join(BASE_DIR, "data", "raw", "capa_comunal.gpkg")
TEMP_SCALER_PATH = os.path.join(BASE_DIR, "models", "scaler_temp_global.pkl")
HIST_SHARES_PATH = os.path.join(BASE_DIR, "data", "interim", "shares_comunales.parquet")
TEMP_REG_PATH = os.path.join(BASE_DIR, "data", "rec_2024_2025", "temperatura_comunal_2024_2025.parquet")
COLUMNS_PATH = os.path.join(BASE_DIR, "data", "rec_2024_2025", "columns.txt")
OUT_DIR = os.path.join(BASE_DIR, "data", "rec_2024_2025", "results", "capas_comunales")

os.makedirs(OUT_DIR, exist_ok=True)

# 2. Parámetros del Modelo
IS_COMUNA = 1  # Trabajamos con comunas ahora
SECTORES = ['I', 'R', 'C', 'P', 'T']
A_PARAM = np.exp(-1.1315)  
B_PARAM = 0.8988
AÑOS_TARGET = [2024, 2025]

# 3. Cargar columnas
columns = []  # Lista vacía para que se guarden los nombres de las columnas
with open("../data/rec_2024_2025/columns.txt", "r") as f:
    for line in f: 
        columns.append(line.strip()) 

# 4. Cargar modelos de red neuronal y scaler de temperatura
model = load_model(MODEL_PATH)
temp_scaler = joblib.load(TEMP_SCALER_PATH)

# 5. Cargar los shares regionales
df_hist_shares = pd.read_parquet(HIST_SHARES_PATH)

# 6. Cargar los shares de temperatura
df_temp_global = pd.read_parquet(TEMP_REG_PATH)
comunas = df_temp_global["comuna"].unique().tolist()

In [13]:
# ==========================================
# CELDA 2: CARGA GEOGRÁFICA Y EVALUACIÓN DE COBERTURA
# ==========================================
print("1. Cargando la capa geográfica y estandarizando coordenadas...")
gdf_mapa = gpd.read_file(LAYER_PATH)

# Asegurarse de usar una proyección en metros (ej. EPSG:32719 UTM 19S para Chile)
# Esto es vital para calcular áreas exactas y distancias métricas
if gdf_mapa.crs != "EPSG:32719":
    gdf_mapa = gdf_mapa.to_crs("EPSG:32719")

# Calcular los centroides geométricos y el área física (en km2)
gdf_mapa['centroide'] = gdf_mapa.geometry.centroid
gdf_mapa['area_km2'] = gdf_mapa.geometry.area / 10**6

def limpiar_texto(texto):
    if pd.isna(texto):
        return texto
    # Normaliza a formato NFKD, codifica a ASCII ignorando caracteres especiales (tildes) y decodifica
    texto_limpio = unicodedata.normalize('NFKD', str(texto)).encode('ASCII', 'ignore').decode('utf-8')
    return texto_limpio.strip().upper()

# 2. Homogeneizar textos para el cruce (mayúsculas y sin espacios extra)
df_hist_shares['comuna'] = df_hist_shares['comuna'].apply(limpiar_texto)
gdf_mapa['comuna'] = gdf_mapa['comuna'].apply(limpiar_texto)
comunas_gdf = gdf_mapa['comuna'].unique().tolist()
comunas_totales_lista = [str(c).strip().upper() for c in comunas_gdf]

# 3. Separar los universos: las 212 con datos y las faltantes
comunas_con_datos = df_hist_shares['comuna'].unique().tolist()
comunas_faltantes = [c for c in comunas_totales_lista if c not in comunas_con_datos]

print(f"Total comunas mapeadas: {len(comunas_totales_lista)}")
print(f"Comunas con datos históricos: {len(comunas_con_datos)}")
print(f"Comunas a extrapolar: {len(comunas_faltantes)}")

1. Cargando la capa geográfica y estandarizando coordenadas...
Total comunas mapeadas: 345
Comunas con datos históricos: 212
Comunas a extrapolar: 133


In [14]:
# ==========================================
# CELDA 3: MOTOR DE EXTRAPOLACIÓN ESPACIAL (IDW + ÁREA)
# ==========================================

warnings.filterwarnings('ignore')

print("2. Iniciando extrapolación espacial (Tamaño y Cercanía)...")

gdf_con_datos = gdf_mapa[gdf_mapa['comuna'].isin(comunas_con_datos)].copy()
gdf_faltantes = gdf_mapa[gdf_mapa['comuna'].isin(comunas_faltantes)].copy()

cols_consumo = [f"consumo_{s}_MWh" for s in SECTORES]
dfs_extrapolados = []

for idx, row in gdf_faltantes.iterrows():
    com_faltante = row['comuna']
    region_faltante = row['region'] if 'region' in row else "SIN_REGION" 
    area_faltante = row['area_km2']
    centroide_faltante = row['centroide']
    
    gdf_con_datos['distancia_m'] = gdf_con_datos['centroide'].distance(centroide_faltante)
    
    vecinos = gdf_con_datos.nsmallest(3, 'distancia_m')
    df_vecinos = df_hist_shares[df_hist_shares['comuna'].isin(vecinos['comuna'])].copy()
    df_vecinos = df_vecinos.merge(vecinos[['comuna', 'area_km2', 'distancia_m']], on='comuna', how='left')
    
    df_vecinos['peso'] = 1.0 / (df_vecinos['distancia_m'] + 1.0) 
    
    # Función reducida: Solo se encarga de la matemática de interpolación espacial
    def estimar_mes(grupo):
        pesos_norm = grupo['peso'] / grupo['peso'].sum()
        fila = {}
        consumo_mensual_total = 0.0
        
        for col in cols_consumo:
            densidad_vecino = grupo[col] / grupo['area_km2']
            densidad_promedio = (densidad_vecino * pesos_norm).sum()
            consumo_est = densidad_promedio * area_faltante
            
            fila[col] = consumo_est
            consumo_mensual_total += consumo_est
            
        fila['consumo_total_mes_MWh'] = consumo_mensual_total
        return pd.Series(fila)

    # 6. Agrupar por año_ms (el reset_index devuelve año_ms como columna)
    df_estimado = df_vecinos.groupby('año_ms').apply(estimar_mes).reset_index()
    
    # === SOLUCIÓN VECTORIZADA DE FECHAS ===
    # Convertimos la columna de forma segura y extraemos como enteros (evita floats)
    df_estimado['año_ms'] = pd.to_datetime(df_estimado['año_ms'], errors='coerce')
    df_estimado['año'] = df_estimado['año_ms'].dt.year.astype('Int64') # Int64 con mayúscula soporta nulos sin pasar a float
    df_estimado['mes'] = df_estimado['año_ms'].dt.month.astype('Int64')
    
    # Agregar meta-tags espaciales de vuelta
    df_estimado['comuna'] = com_faltante
    df_estimado['region'] = region_faltante
    
    dfs_extrapolados.append(df_estimado)

# 7. Unificar ambos mundos en tu Base de Datos Maestra
df_hist_extrapolado = pd.concat(dfs_extrapolados, ignore_index=True)
df_hist_shares["mes"] = df_hist_shares["año_ms"].dt.month
df_shares_completo = pd.concat([df_hist_shares, df_hist_extrapolado], ignore_index=True)

print(f"Extrapolación finalizada con éxito.")
print(f"Dataset consolidado final cuenta con información de {len(df_shares_completo['comuna'].unique())} comunas.")

2. Iniciando extrapolación espacial (Tamaño y Cercanía)...
Extrapolación finalizada con éxito.
Dataset consolidado final cuenta con información de 345 comunas.


### Extrapolación de los shares en el tiempo
TBD

In [15]:
# ==========================================
# CELDA 4: EXTRAPOLACIÓN DE SHARES Y RATIOS (CORREGIDA)
# ==========================================

print("Iniciando extrapolación de consumos y cálculo de shares por mes...")

sectores = ['Industrial', 'Residencial', 'Comercial', 'Público', 'Transporte']
cols_sectores = [f"consumo_{sector}_MWh" for sector in SECTORES]

proyecciones = []
comunas = df_shares_completo["comuna"].unique().tolist()

for comuna in comunas:
    # 1. Aislamos la historia exclusiva de esta comuna y la ordenamos
    df_comuna = df_shares_completo[df_shares_completo['comuna'] == comuna].sort_values(['año', 'mes']).copy()
    
    # Guardamos todo el histórico original de la comuna en la lista
    proyecciones.append(df_comuna)
    
    # 2. Iterar por cada mes único presente en la historia de la comuna
    for mes in df_comuna["mes"].unique().tolist():
        # ¡CORRECCIÓN CRÍTICA!: Filtrar estrictamente el histórico para ESTE mes específico
        df_mes = df_comuna[df_comuna['mes'] == mes]
        years_hist = df_mes['año'].values
        
        # 3. Proyectar para los años objetivo (2024, 2025)
        for target_year in AÑOS_TARGET:
            # Si el año ya existe para este mes en los datos originales, no lo sobreescribimos
            if target_year in years_hist:
                continue
                
            nueva_fila = {'año': target_year, 'comuna': comuna, "mes": mes}
            
            # Preservar la región si existe en el DataFrame base
            if 'region' in df_mes.columns:
                nueva_fila['region'] = df_mes.iloc[0]['region']
                
            for sec in cols_sectores:
                valores_hist = df_mes[sec].values
                if len(years_hist) < 2:
                    # Si no hay historia suficiente para trazar una recta, copiamos el último valor de ese mes
                    nueva_fila[sec] = valores_hist[-1] if len(valores_hist) > 0 else 0.0
                else:
                    # Extrapolación lineal exclusiva para este sector, comuna y mes
                    z = np.polyfit(years_hist, valores_hist, 1)
                    p = np.poly1d(z)
                    # FILTRO FÍSICO: El consumo proyectado nunca puede ser negativo
                    nueva_fila[sec] = max(0.0, p(target_year))
            
            proyecciones.append(pd.DataFrame([nueva_fila]))

# 4. Unir todo el historial + el futuro en un solo DataFrame
df_shares_proyectados = pd.concat(proyecciones, ignore_index=True)

# 5. Calcular los totales y los Shares definitivos por mes y comuna
df_shares_proyectados['consumo_total_mes_MWh'] = df_shares_proyectados[cols_sectores].sum(axis=1)

for sec in SECTORES:
    sec_col = f"consumo_{sec}_MWh"
    nombre_columna = f'share_{sec}'
    
    # Fracción = Sector / Total (usando np.where para evitar división por cero)
    df_shares_proyectados[nombre_columna] = np.where(
        df_shares_proyectados['consumo_total_mes_MWh'] > 0,
        df_shares_proyectados[sec_col] / df_shares_proyectados['consumo_total_mes_MWh'],
        0.0
    )

# 6. Calcular el consumo total nacional agrupado por AÑO y MES
df_nacional = df_shares_proyectados.groupby(['año', 'mes'])['consumo_total_mes_MWh'].sum().reset_index()
df_nacional.rename(columns={'consumo_total_mes_MWh': 'total_consumo_nacional'}, inplace=True)

# ¡CORRECCIÓN CRÍTICA!: Unir el total nacional usando ambas llaves temporales ['año', 'mes']
df_shares_proyectados = pd.merge(df_shares_proyectados, df_nacional, on=['año', 'mes'], how='left')

# 7. Calcular la proporción mensual (total_comunal / total_nacional para cada mes)
df_shares_proyectados['region_comuna_share'] = np.where(
    df_shares_proyectados['total_consumo_nacional'] > 0,
    df_shares_proyectados['consumo_total_mes_MWh'] / df_shares_proyectados['total_consumo_nacional'],
    0.0
)

print("¡Proyección de shares y ratios mensuales completada con éxito!")

Iniciando extrapolación de consumos y cálculo de shares por mes...
¡Proyección de shares y ratios mensuales completada con éxito!


### Cálculo de lags de temperatura y series del calendario comunal
Acá se van a calcular (para cada comuna): 
- Los lags de temperatura que el modelo considera para la "inercia" de $\tau = 7$ horas.
- Series de sin y cos de las horas/semanas/año del intervalo de tiempo que se va a hacer forecast.

En cuanto a los indicadores que definen si es día festivo o no, se determinarán a nivel regional, ya que la librería ``holidays`` de Python tiene integrada sólo la subdivisión de Chile hasta ese nivel de desagregación espacial.



In [16]:
# PROBAR FUNCIONES DE time_features.py
def build_temperature_lags(
        df_temp, 
        temp_col='temperatura', 
        tau=7
):
    """
    Toma un Dataframe anual de temperatura y genera los lags, 
    usando las últimas 'tau' horas para rellenar el inicio.
    """

    df = df_temp.copy()

    for i in range(1, tau + 1): 
        # np.roll desplaza los valores. Al desplazar hacia abajo, 
        # los últimos valores pasan automáticamente al principio
        df[f'temp_t - {i}'] = np.roll(df[temp_col], i)

    return df


def build_calendar_features(df, dt_col='fecha_hora', country='CL', subdiv=None):
    """
    Construye las variables trigonométricas y categóricas del calendario
    a partir de una columna de fecha y hora.
    
    Args:
        df (pd.DataFrame): DataFrame que contiene la serie temporal.
        dt_col (str): Nombre de la columna con las fechas (Datetime).
        country (str): Código ISO del país para buscar los feriados (Defecto: 'CL').
        subdiv (str): Código  ISO de la subdivisón (región) del país (Defecto: None)
        
    Returns:
        pd.DataFrame: DataFrame con las nuevas columnas de features temporales.
    """
    df = df.copy()
    
    # 0. Asegurar que la columna de entrada sea de tipo datetime de Pandas
    if not pd.api.types.is_datetime64_any_dtype(df[dt_col]):
        df[dt_col] = pd.to_datetime(df[dt_col])
        
    # 1. Extraer componentes base
    hour = df[dt_col].dt.hour
    day_of_week = df[dt_col].dt.dayofweek  # Lunes = 0, Domingo = 6
    day_of_year = df[dt_col].dt.dayofyear
    # Detectar años bisiestos para ajustar la longitud del ciclo anual
    days_in_year = df[dt_col].dt.is_leap_year.map({True: 366, False: 365})
    
    # 2. Transformaciones Trigonométricas (Ciclos)
    # Ciclo diario (24 horas)
    df['hour_sin'] = np.sin(2 * np.pi * hour / 24)
    df['hour_cos'] = np.cos(2 * np.pi * hour / 24)
    
    # Ciclo semanal (7 días)
    df['dow_sin'] = np.sin(2 * np.pi * day_of_week / 7)
    df['dow_cos'] = np.cos(2 * np.pi * day_of_week / 7)
    
    # Ciclo anual (365/366 días) - *Sugerido para complementar estacionalidad
    df['doy_sin'] = np.sin(2 * np.pi * day_of_year / days_in_year)
    df['doy_cos'] = np.cos(2 * np.pi * day_of_year / days_in_year)
    
    # 3. Flags Categóricos (Fines de semana y Festivos)
    df['is_weekend'] = df[dt_col].dt.dayofweek.isin([5, 6]).astype(int)
    
    # Obtener festivos del país para los años presentes en el DataFrame
    years = df[dt_col].dt.year.unique()
    cl_holidays = holidays.country_holidays(country, subdiv=subdiv, years=years)
    
    # Mapear si la fecha cae en un día festivo
    df['is_holiday'] = df[dt_col].dt.date.apply(lambda d: d in cl_holidays).astype(int)
    
    # 4. Día laboral (Es True solo si NO es fin de semana y NO es festivo)
    df['is_working_day'] = ((df['is_weekend'] == 0) & (df['is_holiday'] == 0)).astype(int)
    
    return df


In [17]:
regiones_ISO = {
    "TARAPACÁ": "TA", 
    "ANTOFAGASTA": "AN", 
    "ATACAMA": "AT", 
    "COQUIMBO": "CO", 
    "VALPARAÍSO": "VS", 
    "LIBERTADOR GENERAL BERNARDO O'HIGGINS": "LI", 
    "MAULE": "ML", 
    "BIOBÍO": "BI", 
    "LA ARAUCANÍA": "AR", 
    "LOS LAGOS": "LL", 
    "AYSÉN DEL GENERAL CARLOS IBÁÑEZ DEL CAMPO": "AI", 
    "MAGALLANES Y DE LA ANTÁRTICA CHILENA": "MA", 
    "METROPOLITANA DE SANTIAGO": "RM", 
    "LOS RÍOS": "LR", 
    "ARICA Y PARINACOTA": "AP", 
    "ÑUBLE": "NB"
}

gdf_clean = gpd.read_file(LAYER_PATH)

In [18]:
print("Generando lags de temperatura y features de calendario...")

# 1. Asegurar que la columna global sea datetime antes de entrar al bucle
df_temp_global['fecha_hora'] = pd.to_datetime(df_temp_global['fecha_hora'])
comunas_temp = df_temp_global["comuna"].unique().tolist()
# 2. Crear una lista vacía para ir guardando los pedazos de cada región
lista_comunas_procesadas = []

# Bucle for por comunas
for comuna in comunas_temp:
    print(f"Procesando características temporales para: {comuna}")
    region = gdf_clean[gdf_clean["comuna"] == comuna]["region"].values[0]
    print(f"Región: {region}")
    region_ISO = regiones_ISO[region]
    print(f"ISO: {region_ISO}")
    
    # 3. Filtrar los datos de la comuna y ORDENAR cronológicamente (CRÍTICO para np.roll)
    df_com = df_temp_global[df_temp_global["comuna"] == comuna].copy()
    df_com["region"] = region  # asignamos la columna región para cruzar con capa geoespacial 
    df_com = df_com.sort_values(by="fecha_hora").reset_index(drop=True)
    
    # 4. Generar los Lags de temperatura (temp_t-1 a temp_t-7)
    df_com = build_temperature_lags(df_com, temp_col='temperatura', tau=7)
    
    # 5. Generar las variables de calendario (Seno, Coseno, Feriados)
    # Nota: Si tu variable 'region' contiene el código ISO exacto de la región (ej: 'RM', 'AN'), 
    # puedes pasar subdiv=region. Si contiene el nombre completo, usa subdiv=None 
    # para usar los feriados nacionales de Chile ('CL').
    df_com = build_calendar_features(df_com, dt_col='fecha_hora', country='CL', subdiv=region_ISO)
    
    # 6. Almacenar el DataFrame ya procesado en la lista
    lista_comunas_procesadas.append(df_com)

# 7. Unir (Concatenar) todas las regiones procesadas en un solo DataFrame maestro
df_inputs = pd.concat(lista_comunas_procesadas, ignore_index=True)

# 8. Escalar las temperaturas (Si el modelo las espera escaladas)
# Asumiendo que temp_scaler ya fue cargado en la Celda 1
columnas_temp = ['temperatura'] + [f'temp_t - {i}' for i in range(1, 8)]
# Es importante que el scaler reciba las columnas en el mismo orden en que fue entrenado
df_inputs[columnas_temp] = temp_scaler.transform(df_inputs[columnas_temp])

print(f"¡Dataset de inputs temporales creado exitosamente! Shape: {df_inputs.shape}")

Generando lags de temperatura y features de calendario...
Procesando características temporales para: IQUIQUE
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: ALTO HOSPICIO
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: POZO ALMONTE
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: CAMIÑA
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: COLCHANE
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: HUARA
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: PICA
Región: TARAPACÁ
ISO: TA
Procesando características temporales para: ANTOFAGASTA
Región: ANTOFAGASTA
ISO: AN
Procesando características temporales para: MEJILLONES
Región: ANTOFAGASTA
ISO: AN
Procesando características temporales para: SIERRA GORDA
Región: ANTOFAGASTA
ISO: AN
Procesando características temporales para: TALTAL
Región: ANTOFAGASTA
ISO: AN
Procesando características temporales para: CALAMA
Región: ANTOFAG

NOTA: Ñuble no está al 100% en los meses, por lo que hay que extrapolar

In [20]:
# ==========================================
# CELDA 4: FUSIÓN GLOBAL Y MATRIZ DE INPUTS EXACTA   
# ==========================================
print("Iniciando fusión global de shares proyectados y características temporales...")
# Agregar algunas características en df_inputs
df_inputs["año"] = df_inputs["fecha_hora"].dt.year.astype('Int64')
df_inputs["mes"] = df_inputs["fecha_hora"].dt.month.astype('Int64')
if "region_gdf" in df_inputs.columns:
    pass
else: 
    df_inputs["region_gdf"] = df_inputs["region"]
    df_inputs["comuna_gdf"] = df_inputs["comuna"]
    df_inputs['region'] = df_inputs['region'].apply(limpiar_texto)
    df_inputs['comuna'] = df_inputs['comuna'].apply(limpiar_texto)
# Eliminar la columna "año_ms"
if "año_ms" in df_shares_proyectados.columns:
    df_shares_proyectados.drop(columns=["año_ms"], inplace=True) 
df_shares_proyectados["region"] = df_shares_proyectados["region"].apply(limpiar_texto)

# 1. Realizar el merge masivo usando 'año', 'mes', 'region' y 'comuna' como llaves
df_master = pd.merge(df_inputs, df_shares_proyectados, on=['año', 'mes', 'region', 'comuna'], how='left')
df_master.rename(columns={'comuna': 'region_comuna'}, inplace=True)

# 2. Agregar el flag espacial exigido por la arquitectura de la red neuronal global
# IS_COMUNA viene definido como 0 desde la Celda 1
df_master['is_comuna'] = IS_COMUNA

# 3. Control de Calidad: Verificar si hubo pérdidas en el cruce de datos
filas_nulas = df_master['share_R'].isna().sum()
if filas_nulas > 0:
    print(f"⚠️ ¡Advertencia! Hay {filas_nulas} registros horarias que no encontraron su share anual.")
    print("Esto suele pasar si los nombres de las regiones difieren entre las bases (ej. tildes o alias).")
    # Opcional: llenar con 0 o con el promedio si deseas forzar la ejecución
    # df_master = df_master.fillna(0) 
else:
    print("✅ Cruce de datos completado exitosamente. Cero registros nulos encontrados.")

# 4. Validación estructural del orden del Vector de Entrada (Matriz X)
print("\nValidando alineación con la lista exigida por la Red Neuronal (columns.txt)...")

# Buscamos si hay alguna columna en columns.txt que no exista en nuestro DataFrame fusionado
columnas_faltantes = [col for col in columns if col not in df_master.columns]

if columnas_faltantes:
    print(f"❌ ERROR CRÍTICO: El modelo exige columnas que no están en el DataFrame: {columnas_faltantes}")
    print("👉 Consejo: Revisa si hay discrepancias de nombres (ej. 'hour_sin' vs 'sin_hora' o 'temp_t - 1' vs 'temp_t-1').")
    print("Modifica el nombre en el DataFrame para que calce exactamente con lo que pide columns.txt")
else:
    print(f"✅ Alineación perfecta. Las {len(columns)} variables requeridas están presentes.")
    
    # 5. Extracción de la Matriz X Definitiva (Tensor Ciego para Keras)
    # Al pasar la lista 'columns', forzamos a que las columnas queden en el ORDEN EXACTO en que se entrenó la red
    X_inferencia = df_master[columns].values
    
    print(f"\n⚡ ¡Matriz X construida con éxito!")
    print(f"-> Dimensiones finales del input para model.predict(): {X_inferencia.shape}")


Iniciando fusión global de shares proyectados y características temporales...
⚠️ ¡Advertencia! Hay 92136 registros horarias que no encontraron su share anual.
Esto suele pasar si los nombres de las regiones difieren entre las bases (ej. tildes o alias).

Validando alineación con la lista exigida por la Red Neuronal (columns.txt)...
✅ Alineación perfecta. Las 24 variables requeridas están presentes.

⚡ ¡Matriz X construida con éxito!
-> Dimensiones finales del input para model.predict(): (6052680, 24)
